In [18]:
import re
import json


In [1]:
import pandas as pd
import numpy as np

dates = pd.date_range(start='2024-01-01', end='2025-03-31', freq='D')
coffee_types = ['Espresso', 'Latte', 'Cappuccino', 'Americano']

data = {
    'date': np.random.choice(dates, 500),
    'coffee_type': np.random.choice(coffee_types, 500),
    'quantity': np.random.randint(1, 20, 500),
    'revenue': np.random.uniform(3.0, 8.0, 500).round(2)
}

df = pd.DataFrame(data)
df.to_csv('coffee_sales.csv', index=False)
print(df.head())

        date coffee_type  quantity  revenue
0 2024-09-23    Espresso        13     3.14
1 2024-11-12       Latte        18     6.31
2 2024-04-11       Latte         7     5.62
3 2024-01-22   Americano         7     5.73
4 2024-10-03    Espresso        14     7.61


In [8]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()
api_key = os.getenv("OPEN_API_KEY")  
client = OpenAI(api_key=api_key)

In [3]:
print(df.dtypes)
print(df.head(3))


date           datetime64[us]
coffee_type               str
quantity                int64
revenue               float64
dtype: object
        date coffee_type  quantity  revenue
0 2024-09-23    Espresso        13     3.14
1 2024-11-12       Latte        18     6.31
2 2024-04-11       Latte         7     5.62


In [14]:
def generate_chart_code(instruction: str, model: str, out_path_v1: str) -> str:
    prompt = f"""
    You are a data visualization expert.
    Return your answer *strictly* in this format:
    <execute_python>
    # valid python code here
    </execute_python>
    Do not add explanations, only the tags and the code.
    The code should create a visualization from a DataFrame 'df' with these columns:
    - date        (datetime64 — already parsed; use df['date'].dt.year, df['date'].dt.month)
    - coffee_type (string: 'Espresso', 'Latte', 'Cappuccino', 'Americano')
    - quantity    (int — number of units sold)
    - revenue     (float — revenue in dollars)

    User instruction: {instruction}

    Requirements for the code:
    1. Assume the DataFrame is already loaded as 'df'.
    2. Use matplotlib for plotting.
    3. Add clear title, axis labels, and legend if needed.
    4. Save the figure as '{out_path_v1}' with dpi=300.
    5. Do not call plt.show().
    6. Close all plots with plt.close().
    7. Add all necessary import statements.
    8. CRITICAL: 'date' is datetime64 — never use string operations on it.
       Filter by year using df['date'].dt.year, by month using df['date'].dt.month.

    Return ONLY the code wrapped in <execute_python> tags.
    """
    response = client.chat.completions.create(
        model=model,
        max_tokens= 1024,
        messages=[{"role":"user","content":prompt}]
    )

    return response.choices[0].message.content


In [16]:
from IPython.display import display, HTML

def print_html(content: str, title: str = ""):
    display(HTML(f"<h3>{title}</h3><pre style='background:#f4f4f4; padding:1rem; border-radius:6px; white-space:pre-wrap'>{content}</pre>"))
code_v1 = generate_chart_code(
    instruction="Create a plot comparing Q1 coffee sales in 2024 and 2025 using the data in coffee_sales.csv.",
    model="gpt-4o-mini",
    out_path_v1="chart_v1.png"
)
print_html(code_v1, title="LLM output with first draft code")

In [20]:
import re
from IPython.display import display, HTML

def print_html(content: str, title: str = "", is_image: bool = False):
    if is_image:
        display(HTML(f"<h3>{title}</h3><img src='{content}' style='max-width:800px'/>"))
    else:
        display(HTML(f"<h3>{title}</h3><pre style='background:#f4f4f4; padding:1rem; border-radius:6px; white-space:pre-wrap'>{content}</pre>"))

# Extract code from LLM response
match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", code_v1)
if match:
    initial_code = match.group(1).strip()
    print_html(initial_code, title="Extracted Code to Execute")
    exec_globals = {"df": df}
    exec(initial_code, exec_globals)
else:
    print("No <execute_python> tags found in LLM response")

# Display the saved chart
print_html(
    content="chart_v1.png",
    title="Generated Chart (V1)",
    is_image=True
)

In [21]:
import base64
import os

def encode_image_b64(chart_path: str) -> tuple[str, str]:
    # get file extension to determine media type
    ext = os.path.splitext(chart_path)[1].lower()
    media_type_map = {
        ".png": "image/png",
        ".jpg": "image/jpeg",
        ".jpeg": "image/jpeg",
        ".gif": "image/gif",
        ".webp": "image/webp"
    }
    media_type = media_type_map.get(ext, "image/png")
    
    # read image and convert to base64
    with open(chart_path, "rb") as f:  # rb = read binary
        b64 = base64.b64encode(f.read()).decode("utf-8")
    
    return media_type, b64

In [24]:
def ensure_execute_python_tags(code: str) -> str:
    if "<execute_python>" in code:
        return code  # already has tags, leave it
    return f"<execute_python>\n{code}\n</execute_python>" 

In [ ]:
def reflect_on_image_and_regenerate(
    chart_path: str,
    instruction: str,
    model_name: str,
    out_path_v2: str,
    code_v1: str,
) -> tuple[str, str]:

    media_type, b64 = encode_image_b64(chart_path)

    prompt = f"""
    You are a data visualization expert.
    Critique the attached chart and the original code, then return improved code.

    Original code:
    {code_v1}

    OUTPUT FORMAT (STRICT):
    1) First line only: a valid JSON object with ONLY the "feedback" field.
    Example: {{"feedback": "The legend is unclear and axis labels overlap."}}

    2) After a newline, the improved code wrapped in:
    <execute_python>
    ...
    </execute_python>

    HARD CONSTRAINTS:
    - No markdown, no backticks, no extra text outside the two parts above.
    - Use matplotlib only (no seaborn).
    - df is already loaded — do not read from files.
    - Save to '{out_path_v2}' with dpi=300.
    - Call plt.close() at the end.
    - Include all necessary imports.
    - 'date' is datetime64 — use df['date'].dt.year to filter by year.

    Schema:
    - date        (datetime64)
    - coffee_type (string: Espresso, Latte, Cappuccino, Americano)
    - quantity    (int)
    - revenue     (float)

    Instruction: {instruction}
    """

    response = client.chat.completions.create(
        model=model_name,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:{media_type};base64,{b64}"}
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }]
    )
    content = response.choices[0].message.content

    # parse feedback JSON from first line
    lines = content.strip().splitlines()
    json_line = lines[0].strip() if lines else ""
    try:
        obj = json.loads(json_line)
    except Exception as e:
        m_json = re.search(r"\{.*?\}", content, flags=re.DOTALL)
        if m_json:
            try:
                obj = json.loads(m_json.group(0))
            except Exception as e2:
                obj = {"feedback": f"Failed to parse JSON: {e2}"}
        else:
            obj = {"feedback": f"Failed to find JSON: {e}"}

    # extract refined code
    m_code = re.search(r"<execute_python>([\s\S]*?)</execute_python>", content)
    refined_code_body = m_code.group(1).strip() if m_code else ""
    refined_code = ensure_execute_python_tags(refined_code_body)

    feedback = str(obj.get("feedback", "")).strip()
    return feedback, refined_code








    

In [25]:
# Generate feedback alongside reflected code
feedback, code_v2 = reflect_on_image_and_regenerate(
    chart_path="chart_v1.png",            
    instruction="Create a plot comparing Q1 coffee sales in 2024 and 2025 using the data in coffee_sales.csv.", 
    model_name="o4-mini",
    out_path_v2="chart_v2.png",
    code_v1=code_v1,   # pass in the original code for context        
)

print_html(feedback, title="Feedback on V1 Chart")
print_html(code_v2, title="Regenerated Code Output (V2)")

In [26]:
# Get the code within the <execute_python> tags
match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", code_v2)
if match:
    reflected_code = match.group(1).strip()
    exec_globals = {"df": df}
    exec(reflected_code, exec_globals)

# If code run successfully, the file chart_v2.png should have been generated
print_html(
    content="chart_v2.png",
    title="Regenerated Chart (V2)",
    is_image=True
)

In [27]:
def load_and_prepare_data(dataset_path: str):
    df = pd.read_csv(dataset_path)
    df['date'] = pd.to_datetime(df['date'])
    return df

def run_workflow(
    dataset_path: str,
    user_instructions: str,
    generation_model: str,
    reflection_model: str,
    image_basename: str = "chart",
):
    # 0) load dataset
    df = load_and_prepare_data(dataset_path)
    print_html(df.sample(n=5).to_html(), title="Random Sample of Dataset")

    # paths
    out_v1 = f"{image_basename}_v1.png"
    out_v2 = f"{image_basename}_v2.png"

    # 1) generate v1 code
    print_html("Step 1: Generating chart code (V1)... 📈")
    code_v1 = generate_chart_code(
        instruction=user_instructions,
        model=generation_model,
        out_path_v1=out_v1,
    )
    print_html(code_v1, title="LLM output with first draft code (V1)")

    # 2) execute v1
    print_html("Step 2: Executing chart code (V1)... 💻")
    match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", code_v1)
    if match:
        initial_code = match.group(1).strip()
        exec(initial_code, {"df": df})
    print_html(out_v1, is_image=True, title="Generated Chart (V1)")

    # 3) reflect on v1 → get feedback + v2 code
    print_html("Step 3: Reflecting on V1 and generating improvements... 🔁")
    feedback, code_v2 = reflect_on_image_and_regenerate(
        chart_path=out_v1,
        instruction=user_instructions,
        model_name=reflection_model,
        out_path_v2=out_v2,
        code_v1=code_v1,
    )
    print_html(feedback, title="Reflection feedback on V1")
    print_html(code_v2, title="LLM output with revised code (V2)")

    # 4) execute v2
    print_html("Step 4: Executing refined chart code (V2)... 🖼️")
    match = re.search(r"<execute_python>([\s\S]*?)</execute_python>", code_v2)
    if match:
        reflected_code = match.group(1).strip()
        exec(reflected_code, {"df": df})
    print_html(out_v2, is_image=True, title="Regenerated Chart (V2)")

    return {
        "code_v1": code_v1,
        "chart_v1": out_v1,
        "feedback": feedback,
        "code_v2": code_v2,
        "chart_v2": out_v2,
    }

In [31]:

results = run_workflow(
    dataset_path="coffee_sales.csv",
    user_instructions="Create a plot comparing Q1 coffee sales in 2024 and 2025.",
    generation_model="gpt-4o-mini",
    reflection_model="gpt-4o-mini",
    image_basename="chart"
)

,date,coffee_type,quantity,revenue
12,2024-09-23,Americano,3,4.07
173,2025-01-11,Latte,6,4.04
71,2025-01-13,Americano,19,3.30
367,2024-10-03,Cappuccino,16,3.78
335,2024-10-14,Cappuccino,6,3.93
